# Session 9 — Markov Chains: Long-Run Behaviour

Every example from the session notes, worked in Python. Page numbers refer to the session notes.

- **SymPy** solves the linear systems exactly, so the answers come out as fractions, as in the notes.
- **NumPy** is used for matrix powers and simulation.

**Watch out:** in SymPy `P**n` is the matrix power. In NumPy `P**n` raises every *entry* to the
power n. In NumPy always use `P @ P` or `np.linalg.matrix_power(P, n)`.

In [1]:
import numpy as np
import sympy as sp
from sympy import Rational as R

np.set_printoptions(precision=4, suppress=True)
rng = np.random.default_rng(9)

## 1. What happens after many steps? (page 1)

The education chain (Session 8), order (H, S, U), from each of the three starting states.
Row i of $P^{n}$ is the distribution of $X_n$ when the chain starts in state i.

In [2]:
P_edu = np.array([[0.8, 0.1, 0.1],
                  [0.2, 0.6, 0.2],
                  [0.25, 0.25, 0.5]])

for n in [1, 2, 4, 8, 16]:
    print(f"n = {n:2d}: P(X_n = H) from H, S, U =", np.linalg.matrix_power(P_edu, n)[:, 0])

n =  1: P(X_n = H) from H, S, U = [0.8  0.2  0.25]
n =  2: P(X_n = H) from H, S, U = [0.685 0.33  0.375]
n =  4: P(X_n = H) from H, S, U = [0.5799 0.458  0.4778]
n =  8: P(X_n = H) from H, S, U = [0.5325 0.5184 0.5209]
n = 16: P(X_n = H) from H, S, U = [0.5264 0.5262 0.5262]


## 2. Communicating classes (page 2)

The five-state chain. State j is accessible from i if $p^{(n)}_{ij} > 0$ for some $n \ge 0$.
With r states it is enough to check $n = 0, 1, \dots, r-1$: a shortest route never needs more than
$r - 1$ steps.

In [3]:
P5 = np.array([[0.5, 0.5, 0.0, 0.0, 0.0],
               [0.3, 0.0, 0.4, 0.0, 0.3],
               [0.0, 0.0, 0.4, 0.6, 0.0],
               [0.0, 0.0, 1.0, 0.0, 0.0],
               [0.0, 0.0, 0.0, 0.0, 1.0]])
r = P5.shape[0]

# reach[i, j] = True if j is accessible from i (i -> j)
reach = sum(np.linalg.matrix_power(P5, n) for n in range(r)) > 0
communicate = reach & reach.T          # i <-> j

states = [1, 2, 3, 4, 5]
classes = []
for i in range(r):
    cls = {states[j] for j in range(r) if communicate[i, j]}
    if cls not in classes:
        classes.append(cls)

print("1 -> 3?", reach[0, 2], "   3 -> 1?", reach[2, 0])
print("classes:", classes)

1 -> 3? True    3 -> 1? False
classes: [{1, 2}, {3, 4}, {5}]


## 3. Closed classes, recurrent and transient states (page 4)

A class is closed if no arrow leaves it. In a finite chain: closed = recurrent, can be left = transient.

In [4]:
for cls in classes:
    idx = [s - 1 for s in cls]
    out = [j for i in idx for j in range(r) if P5[i, j] > 0 and j not in idx]
    print(cls, "closed (recurrent)" if not out else "can be left (transient)")

{1, 2} can be left (transient)
{3, 4} closed (recurrent)
{5} closed (recurrent)


**$f_{11}$ by first-step analysis.** Let $h_k = P(\text{the chain ever reaches } 1 \mid X_0 = k)$
with $h_1 = 1$. From 3, 4 and 5 the chain cannot reach 1, so $h_3 = h_4 = h_5 = 0$. Then one step
from 1: $f_{11} = p_{11} + \sum_{k \ne 1} p_{1k} h_k$.

In [5]:
h2 = sp.symbols('h2')
h2_val = sp.solve(sp.Eq(h2, R(3, 10)*1 + R(4, 10)*0 + R(3, 10)*0), h2)[0]
f11 = R(1, 2) + R(1, 2)*h2_val
print("h_2 =", h2_val, "   f_11 =", f11, "=", float(f11))

h_2 = 3/10    f_11 = 13/20 = 0.65


## 4. Irreducible or not? (page 5)

A chain is irreducible if every state is accessible from every other state.

In [6]:
def is_irreducible(P):
    r = P.shape[0]
    reach = sum(np.linalg.matrix_power(P, n) for n in range(r)) > 0
    return bool(reach.all())

P_c = np.array([[0, 1, 0],
                [0, 0.5, 0.5],
                [1, 0, 0]])

print("a) education chain:", is_irreducible(P_edu))
print("b) five-state chain:", is_irreducible(P5))
print("c) cycle 1 -> 2 -> 3 -> 1:", is_irreducible(P_c))

a) education chain: True
b) five-state chain: False
c) cycle 1 -> 2 -> 3 -> 1: True


## 5. Absorption probabilities (page 6)

States 1 and 3 are absorbing. $a_i = P(\text{absorption in } 1 \mid X_0 = i)$ with $a_1 = 1$,
$a_3 = 0$, and $a_i = \sum_k p_{ik} a_k$ for the transient states 0 and 2.

In [7]:
a0, a2 = sp.symbols('a0 a2')
a1, a3 = 1, 0

eqs = [sp.Eq(a0, R(1, 5)*a0 + R(1, 5)*a1 + R(1, 5)*a2 + R(2, 5)*a3),
       sp.Eq(a2, R(1, 2)*a1 + R(1, 2)*a3)]
sol = sp.solve(eqs, [a0, a2])
print(sol, "   absorption in 3 from 0:", 1 - sol[a0])

{a0: 3/8, a2: 1/2}    absorption in 3 from 0: 5/8


## 6. A closed class as one absorbing state (page 7)

The five-state chain: merge the closed class {3, 4} into one absorbing state R.
$a_R = 1$, $a_5 = 0$.

In [8]:
a1, a2 = sp.symbols('a1 a2')
aR, a5 = 1, 0

eqs = [sp.Eq(a1, R(1, 2)*a1 + R(1, 2)*a2),
       sp.Eq(a2, R(3, 10)*a1 + R(4, 10)*aR + R(3, 10)*a5)]
sol = sp.solve(eqs, [a1, a2])
print(sol, "   absorbed in 5 from 1:", 1 - sol[a1])

{a1: 4/7, a2: 4/7}    absorbed in 5 from 1: 3/7


## 7. Expected absorption time (pages 8–9)

The walk on {0, 1, 2, 3}; 0 and 3 are absorbing. **Simulate first:** start in 1 and count the steps
until the chain is in A = {0, 3}.

In [9]:
P_walk = np.array([[1, 0, 0, 0],
                   [1/3, 0, 2/3, 0],
                   [0, 1/2, 0, 1/2],
                   [0, 0, 0, 1]])

rng = np.random.default_rng(9)
runs = 20_000
T = np.empty(runs, int)
end_in_3 = np.empty(runs, bool)
for k in range(runs):
    x, n = 1, 0
    while x in (1, 2):
        x = rng.choice(4, p=P_walk[x])
        n += 1
    T[k], end_in_3[k] = n, (x == 3)

print("simulated mean absorption time:", T.mean())
print("simulated P(absorbed in 3):", end_in_3.mean())

simulated mean absorption time: 2.51895
simulated P(absorbed in 3): 0.49835


**Exact:** $t_i = 0$ for $i \in A$ and $t_i = 1 + \sum_k p_{ik} t_k$ for $i \notin A$.

In [10]:
t1, t2 = sp.symbols('t1 t2')
t0, t3 = 0, 0
sol_t = sp.solve([sp.Eq(t1, 1 + R(1, 3)*t0 + R(2, 3)*t2),
                  sp.Eq(t2, 1 + R(1, 2)*t1 + R(1, 2)*t3)], [t1, t2])
print("t:", sol_t)

t: {t1: 5/2, t2: 9/4}


**Absorbed in 3?** Same method with $b_3 = 1$, $b_0 = 0$, where $b_i = P(\text{absorption in } 3 \mid X_0 = i)$.

In [11]:
b1, b2 = sp.symbols('b1 b2')
b0, b3 = 0, 1
sol_b = sp.solve([sp.Eq(b1, R(1, 3)*b0 + R(2, 3)*b2),
                  sp.Eq(b2, R(1, 2)*b1 + R(1, 2)*b3)], [b1, b2])
print("b:", sol_b)

b: {b1: 1/2, b2: 3/4}


**The same systems as a matrix with `rref`.** Move the unknowns to the left: each row is one equation,
the last column is the right-hand side.

In [12]:
# t1 - 2/3 t2 = 1,  -1/2 t1 + t2 = 1
print(sp.Matrix([[1, -R(2, 3), 1], [-R(1, 2), 1, 1]]).rref()[0])
# b1 - 2/3 b2 = 0,  -1/2 b1 + b2 = 1/2
print(sp.Matrix([[1, -R(2, 3), 0], [-R(1, 2), 1, R(1, 2)]]).rref()[0])

Matrix([[1, 0, 5/2], [0, 1, 9/4]])
Matrix([[1, 0, 1/2], [0, 1, 3/4]])


## 8. Mean hitting and return times (page 11)

$t_k$ = mean hitting time of state 1 from k, so $t_1 = 0$. Then $r_1 = 1 + \sum_k p_{1k} t_k$.

In [13]:
P_ret = sp.Matrix([[R(1, 2), R(1, 2), 0],
                   [0, R(1, 3), R(2, 3)],
                   [R(1, 2), R(1, 2), 0]])

t2, t3 = sp.symbols('t2 t3')
t1 = 0
sol = sp.solve([sp.Eq(t2, 1 + P_ret[1, 0]*t1 + P_ret[1, 1]*t2 + P_ret[1, 2]*t3),
                sp.Eq(t3, 1 + P_ret[2, 0]*t1 + P_ret[2, 1]*t2 + P_ret[2, 2]*t3)], [t2, t3])
t = [t1, sol[t2], sol[t3]]
r1 = 1 + sum(P_ret[0, k]*t[k] for k in range(3))
print("t =", t, "   r_1 =", r1)

t = [0, 5, 7/2]    r_1 = 7/2


## 9. Stationary distribution (pages 12–15)

**The method.** $\pi = \pi P$ gives one equation per state, $\pi_j = \sum_k \pi_k p_{kj}$
(column j of P). Drop one of them and add $\pi_1 + \dots + \pi_r = 1$.

In matrix form, transposing P turns the columns into rows:
$(P^{\mathsf T} - I)\,\pi^{\mathsf T} = 0$.

1. Transpose P (the columns become the balance equations).
2. Subtract the identity.
3. Drop the last equation.
4. Add a column of zeros (the right-hand side).
5. Add a row of ones (the sum is 1), then `rref`.

### Two states (page 13)

$P = \begin{bmatrix} 1-a & a \\ b & 1-b \end{bmatrix}$. Balance: flow 1 → 2 = flow 2 → 1, i.e. $\pi_1 a = \pi_2 b$.

In [14]:
a, b, p1, p2 = sp.symbols('a b pi1 pi2', positive=True)
sol = sp.solve([sp.Eq(p1*a, p2*b), sp.Eq(p1 + p2, 1)], [p1, p2])
print(sol)
print("weather chain, a = 0.6, b = 0.3:", {k: v.subs({a: R(6, 10), b: R(3, 10)}) for k, v in sol.items()})

{pi1: b/(a + b), pi2: a/(a + b)}
weather chain, a = 0.6, b = 0.3: {pi1: 1/3, pi2: 2/3}


### Education chain (page 14)

The notes write one equation per state, drop the equation for $\pi_U$, add the sum and state the
solution. Here SymPy solves the same three equations; the five steps are shown for the operator
chain below.

In [15]:
pH, pS, pU = sp.symbols('pi_H pi_S pi_U')
sol = sp.solve([sp.Eq(pH, R(8, 10)*pH + R(2, 10)*pS + R(1, 4)*pU),
                sp.Eq(pS, R(1, 10)*pH + R(6, 10)*pS + R(1, 4)*pU),
                sp.Eq(pH + pS + pU, 1)], [pH, pS, pU])
print(sol, [float(v) for v in sol.values()])

{pi_H: 10/19, pi_S: 5/19, pi_U: 4/19} [0.5263157894736842, 0.2631578947368421, 0.21052631578947367]


### Mobile operators (page 15): the five steps

In [16]:
P_op = sp.Matrix([[R(7, 10), R(2, 10), R(1, 10)],
                  [R(5, 10), R(4, 10), R(1, 10)],
                  [R(6, 10), R(2, 10), R(2, 10)]])
r = P_op.shape[0]

# Step 1: transpose P (the columns become the balance equations)
X = P_op.T

# Step 2: subtract the identity: (P^T - I) pi = 0
X = X - sp.eye(r)

# Step 3: drop the last equation
X = X[0:-1, :]

# Step 4: add a column of zeros (the right-hand side)
X = X.row_join(sp.zeros(r - 1, 1))

# Step 5: add the row of ones (pi_A + pi_B + pi_C = 1), then rref
X = X.col_join(sp.ones(1, r + 1))
X.rref()[0]

Matrix([
[1, 0, 0, 23/36],
[0, 1, 0,   1/4],
[0, 0, 1,   1/9]])

In [17]:
# In one go:
(P_op.T - sp.eye(r))[0:-1, :].row_join(sp.zeros(r - 1, 1)).col_join(sp.ones(1, r + 1)).rref()[0]

Matrix([
[1, 0, 0, 23/36],
[0, 1, 0,   1/4],
[0, 0, 1,   1/9]])

**The same in NumPy.** Replace the last equation by the row of ones and solve, or keep all r
equations plus the sum and use least squares (no equation has to be dropped).

In [18]:
P_op_np = np.array(P_op, dtype=float)
r = P_op_np.shape[0]

A = P_op_np.T - np.eye(r)
A[-1, :] = 1
pi = np.linalg.solve(A, np.r_[np.zeros(r - 1), 1])
print("solve:", pi)

A = np.vstack([P_op_np.T - np.eye(r), np.ones(r)])
pi = np.linalg.lstsq(A, np.r_[np.zeros(r), 1], rcond=None)[0]
print("lstsq:", pi)

print("check pi @ P:", pi @ P_op_np)
print("Session 8, P^3:\n", np.linalg.matrix_power(P_op_np, 3))

solve: [0.6389 0.25   0.1111]
lstsq: [0.6389 0.25   0.1111]
check pi @ P: [0.6389 0.25   0.1111]
Session 8, P^3:
 [[0.641 0.248 0.111]
 [0.633 0.256 0.111]
 [0.64  0.248 0.112]]


### The walk with two absorbing states (page 15)

All solutions of $\pi = \pi P$, $\sum_j \pi_j = 1$: the solution is not unique.

In [19]:
P_walk_s = sp.Matrix([[1, 0, 0, 0],
                      [R(1, 3), 0, R(2, 3), 0],
                      [0, R(1, 2), 0, R(1, 2)],
                      [0, 0, 0, 1]])
pi0, pi1, pi2, pi3 = sp.symbols('pi0 pi1 pi2 pi3')
pi_vec = sp.Matrix([[pi0, pi1, pi2, pi3]])
eqs = list(pi_vec*P_walk_s - pi_vec) + [pi0 + pi1 + pi2 + pi3 - 1]
print(sp.solve(eqs, [pi1, pi2, pi3]))     # pi0 = alpha is free

{pi1: 0, pi2: 0, pi3: 1 - pi0}


## 10. Periodicity (page 16)

$d(i) = \gcd\{n \ge 1 : p^{(n)}_{ii} > 0\}$. Here we look at $n = 1, \dots, 20$.

In [20]:
from math import gcd
from functools import reduce

def period(P, i, n_max=20):
    returns = [n for n in range(1, n_max + 1) if np.linalg.matrix_power(P, n)[i, i] > 1e-12]
    return reduce(gcd, returns), returns[:6]

P_tri = np.array([[0, 1, 0],
                  [0.5, 0, 0.5],
                  [1, 0, 0]])                # a) no self-loops
P_flip = np.array([[0, 1], [1, 0]])         # b) the flip chain

print("a) no self-loops, state 1:", period(P_tri, 0))
print("b) flip chain, state 1:", period(P_flip, 0))

a) no self-loops, state 1: (1, [2, 3, 4, 5, 6, 7])
b) flip chain, state 1: (2, [2, 4, 6, 8, 10, 12])


## 11. Stationary versus limiting (pages 17–18)

**The Ehrenfest model.** 3 molecules in two boxes; each step one molecule, chosen at random, moves to
the other box. State = number of molecules in the left box. The chain has period 2, a stationary
distribution, but no limiting distribution.

In [21]:
P_E = sp.Matrix([[0, 1, 0, 0],
                 [R(1, 3), 0, R(2, 3), 0],
                 [0, R(2, 3), 0, R(1, 3)],
                 [0, 0, 1, 0]])
r = P_E.shape[0]
X = (P_E.T - sp.eye(r))[0:-1, :].row_join(sp.zeros(r - 1, 1)).col_join(sp.ones(1, r + 1))
pi_E = X.rref()[0][:, -1].T
print("pi =", pi_E)

P_E_np = np.array(P_E, dtype=float)
print("p^(n)_00 for n = 1..12:", [round(float(np.linalg.matrix_power(P_E_np, n)[0, 0]), 4) for n in range(1, 13)])

pi = Matrix([[1/8, 3/8, 3/8, 1/8]])
p^(n)_00 for n = 1..12: [0.0, 0.3333, 0.0, 0.2593, 0.0, 0.251, 0.0, 0.2501, 0.0, 0.25, 0.0, 0.25]


The **bus/cycle chain** is aperiodic (self-loop at C), and every row of $P^{n}$ converges to
$\pi = (1/3, 2/3)$, the same $\pi$ as the weather chain.

In [22]:
P_bus = np.array([[0, 1], [0.5, 0.5]])     # order (B, C), self-loop at C

for n in [1, 2, 3, 20, 21]:
    print(f"n = {n:2d}   Ehrenfest p00 = {np.linalg.matrix_power(P_E_np, n)[0, 0]:.4f}"
          f"   bus/cycle row B = {np.linalg.matrix_power(P_bus, n)[0]}")

n =  1   Ehrenfest p00 = 0.0000   bus/cycle row B = [0. 1.]
n =  2   Ehrenfest p00 = 0.3333   bus/cycle row B = [0.5 0.5]
n =  3   Ehrenfest p00 = 0.0000   bus/cycle row B = [0.25 0.75]
n = 20   Ehrenfest p00 = 0.2500   bus/cycle row B = [0.3333 0.6667]
n = 21   Ehrenfest p00 = 0.0000   bus/cycle row B = [0.3333 0.6667]


**$r_0 = 1/\pi_0$ without aperiodicity (page 19).** Hitting times of 0, then the return time.

In [23]:
t1, t2, t3 = sp.symbols('t1 t2 t3')
t0 = 0
sol = sp.solve([sp.Eq(t1, 1 + R(1, 3)*t0 + R(2, 3)*t2),
                sp.Eq(t2, 1 + R(2, 3)*t1 + R(1, 3)*t3),
                sp.Eq(t3, 1 + t2)], [t1, t2, t3])
print(sol, "   r_0 = 1 + t_1 =", 1 + sol[t1], "   1/pi_0 =", 1/pi_E[0])

{t1: 7, t2: 9, t3: 10}    r_0 = 1 + t_1 = 8    1/pi_0 = 8


## 12. Check $r_1 = 1/\pi_1$ (page 20)

The chain from the return-time example (Section 8 above).

In [24]:
r = P_ret.shape[0]
X = (P_ret.T - sp.eye(r))[0:-1, :].row_join(sp.zeros(r - 1, 1)).col_join(sp.ones(1, r + 1))
pi_ret = X.rref()[0][:, -1].T
print("pi =", pi_ret, "   1/pi_1 =", 1/pi_ret[0], "   r_1 =", r1)

pi = Matrix([[2/7, 3/7, 2/7]])    1/pi_1 = 7/2    r_1 = 7/2


## 13. Long-run proportions by simulation (pages 20–21)

One long path of the education chain started in U: the proportion of time in each state.

In [25]:
rng = np.random.default_rng(19)
n_steps = 100_000
cum = P_edu.cumsum(axis=1)
u = rng.random(n_steps)
x = np.empty(n_steps + 1, int)
x[0] = 2                                   # start in U
for k in range(n_steps):
    x[k + 1] = (u[k] > cum[x[k]]).sum()   # next state from the row of the current state

print("proportions:", np.bincount(x, minlength=3) / len(x))
print("pi:         ", np.array([10, 5, 4]) / 19)

proportions: [0.5271 0.2626 0.2103]
pi:          [0.5263 0.2632 0.2105]


**A set of states.** Proportion of time not unskilled: $\pi_H + \pi_S$.

**What if?** Weather chain with $P(\text{rain} \mid \text{rain})$ raised from 0.4 to 0.5.

**Does $\pi^{(0)}$ matter?** Absorption from a random start: $\sum_i \pi^{(0)}_i a_i$ for the chain
with $a_0 = 3/8$, $a_2 = 1/2$ (Section 5) and $\pi^{(0)} = (1/2, 0, 1/2, 0)$.

In [26]:
print("not unskilled:", R(10, 19) + R(5, 19))

a_new, b_new = R(5, 10), R(3, 10)
print("weather, pi_rain:", b_new/(a_new + b_new), "(was", R(3, 10)/(R(6, 10) + R(3, 10)), ")")

print("absorption in 1 from pi^(0):", R(1, 2)*R(3, 8) + R(1, 2)*R(1, 2))

not unskilled: 15/19
weather, pi_rain: 3/8 (was 1/3 )
absorption in 1 from pi^(0): 7/16


In [25]:
import numpy as np

P = np.array([[0.4, 0.6],     # weather chain, order (rain, no rain)
              [0.3, 0.7]])

starts = {"rain today    (1, 0)    ": np.array([1.0, 0.0]),
          "no rain today (0, 1)    ": np.array([0.0, 1.0]),
          "coin toss     (0.5, 0.5)": np.array([0.5, 0.5])}

for n in [1, 2, 3, 5, 10]:
    Pn = np.linalg.matrix_power(P, n)
    print(f"n = {n}")
    for name, pi0 in starts.items():
        print("  ", name, pi0 @ Pn)      # pi^(n) = pi^(0) P^n

# Check the closed formula P(X_n = rain) = 1/3 + (p - 1/3) * 0.1**n
p = 1.0
for n in [1, 2, 3, 5]:
    exact = (np.array([p, 1 - p]) @ np.linalg.matrix_power(P, n))[0]
    print(n, exact, 1/3 + (p - 1/3) * 0.1**n)

n = 1
   rain today    (1, 0)     [0.4 0.6]
   no rain today (0, 1)     [0.3 0.7]
   coin toss     (0.5, 0.5) [0.35 0.65]
n = 2
   rain today    (1, 0)     [0.34 0.66]
   no rain today (0, 1)     [0.33 0.67]
   coin toss     (0.5, 0.5) [0.335 0.665]
n = 3
   rain today    (1, 0)     [0.334 0.666]
   no rain today (0, 1)     [0.333 0.667]
   coin toss     (0.5, 0.5) [0.3335 0.6665]
n = 5
   rain today    (1, 0)     [0.3333 0.6667]
   no rain today (0, 1)     [0.3333 0.6667]
   coin toss     (0.5, 0.5) [0.3333 0.6667]
n = 10
   rain today    (1, 0)     [0.3333 0.6667]
   no rain today (0, 1)     [0.3333 0.6667]
   coin toss     (0.5, 0.5) [0.3333 0.6667]
1 0.4 0.4
2 0.34 0.33999999999999997
3 0.33399999999999996 0.33399999999999996
5 0.33333999999999997 0.33333999999999997


In [33]:
pi = sp.Matrix([[0.1, 0.9]])
P = sp.Matrix([[0.4, 0.6],
              [0.3, 0.7]])
pi * P**11

Matrix([[0.333333333331, 0.666666666669]])